In [11]:
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.abstract_event_listener import AbstractEventListener
from selenium.webdriver.support.events import EventFiringWebDriver, AbstractEventListener
from selenium.webdriver import ActionChains
from selenium.webdriver.common.keys import Keys
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import UnexpectedAlertPresentException
from selenium import webdriver
from webdriver_auto_update.chrome_app_utils import ChromeAppUtils
from webdriver_auto_update.webdriver_manager import WebDriverManager
import traceback

#* setup
def setup_chrome():
    options = Options()
    options.add_experimental_option("debuggerAddress", "localhost:8989")
    driver = webdriver.Chrome(service=Service(r'C:\bin\chromedriver.exe'), options=options)
    return driver

def get_tabs():
    global merged_dict
    try:
        # if parent.winfo_exists():
        if True:
            print("รายงานจำนวนtabs")

            # * เก็บชื่อ title และ value ของ tab ที่เปิดอยู่
            title_list = []
            # title_list_Idx = [] #!เหมือนจะไม่ได้ใช้
            value_list = []
            # title_dict = {} #!เหมือนจะไม่ได้ใช้
            for idx, handle in enumerate(driver.window_handles):
                driver.switch_to.window(handle)
                # title_list_Idx.append(
                #     driver.title + "["+str(idx)+"]") #!เหมือนจะไม่ได้ใช้
                title_list.append(driver.title)

                value_list.append(driver.current_window_handle)
                # title_dict.update(
                #     {driver.title: driver.current_window_handle}) #!เหมือนจะไม่ได้ใช้

            # * เอาtitle มาทำให้ unique เพราะ title จะสามารถที่จะซ้ำกันได้
            unique_titles = []
            counter = {}
            for item in title_list:
                if item in counter:
                    counter[item] += 1
                    print("counter[item] คือไร: ", counter[item])
                    unique_titles.append(
                        f"{item}{counter[item]-1}")
                else:
                    counter[item] = 1
                    unique_titles.append(item)

            # * เอาList มารวมกัน
            merged_dict = dict(zip(unique_titles, value_list))
            print("มี tabs ไรบ้าง", merged_dict)
            
    except Exception as e:
        traceback_str = traceback.format_exc()
        print(f"An error occirred: {e}")
        print(traceback_str)
        
def fill_items(array_items=[]):
    sku_input_xpath = '/html/body/div[1]/div[2]/div[2]/div[2]/div[1]/div[1]/from/div/div/div[1]/div[1]/span/input'
    
    for item in array_items:
        driver.find_element(By.XPATH, sku_input_xpath).clear()
        driver.find_element(By.XPATH, sku_input_xpath).send_keys(item)
        driver.find_element(By.XPATH, sku_input_xpath).send_keys(Keys.ENTER)


driver = setup_chrome()
get_tabs()
driver.switch_to.window(merged_dict['SMCO :: เปิดการขาย'])
#* เอา function ที่ต้องการเทสมาใส่ข้างล่างนี่
item = ["CO6-010714", "CO6-010334"]
fill_items(item)


รายงานจำนวนtabs
counter[item] คือไร:  2
มี tabs ไรบ้าง {'DevTools': 'CF9036803B7DCF06F2A42AE4340576DB', 'DevTools1': 'DE717BB1E67FB51A8252B7C12C624B57', 'Seller Centre': 'B07483631DF8AC44761BDCE8FCBC8158', 'SMCO :: ลูกค้า': '848A7430A65F0DAC288FFAAA167FABCE', 'SMCO :: เปิดการขาย': '3208D062631D87DF9A099EC599C892FE'}


### PDF READER

In [40]:
# # importing required classes 
# from pypdf import PdfReader 

# # creating a pdf reader object 
# reader = PdfReader('TRB018324080800011-Tranfer.pdf') 

# # printing number of pages in pdf file 
# print(len(reader.pages)) 

# # creating a page object 
# page = reader.pages[0] 

# # extracting text from page 
# print(page.extract_text()) 

from pypdf import PdfReader
import pandas as pd
import re
from openpyxl import load_workbook

extracted_txt:str =""
target_dir = r"C:\Users\ONLINE_MIS\Downloads\TRB018324070800041-Tranfer.pdf"
reader = PdfReader(target_dir)

#* สกัดเอา ข้อความออกมาจากไฟล์
for page in reader.pages:
    extracted_txt += page.extract_text()
    
print(extracted_txt)

#* สกัดเอาค่าที่จำเป็นออกจากข้อความทั้งหมด
#* Regular expression สำหรับการจับ SKU
sku_pattern = r'\b[A-Z]{2}\d-\d{6}\b'

# *Regular expression สำหรับการจับ serial numbers
serial_pattern = r'Shipped\s*\n\n\s*(.*?)Serial\s*:'

#* สกัด SKU
sku_matches = re.findall(sku_pattern, extracted_txt)

#* สกัด serial numbers
serial_matches = re.findall(serial_pattern, extracted_txt, re.DOTALL)

#* จัดการ serial numbers ให้เป็น list ของแต่ละ SKU
serial_numbers_grouped = [serial.strip().replace('\n', '').replace(' ', '').split(',') for serial in serial_matches]

# สร้าง DataFrame ที่แต่ละคอลัมน์เป็น SKU และแต่ละ row เป็น serial number
df = pd.DataFrame.from_dict({sku: serials for sku, serials in zip(sku_matches, serial_numbers_grouped)}, orient='index').transpose()

# โหลดไฟล์ Excel ที่มีอยู่แล้ว
output_excel = r"C:\Users\ONLINE_MIS\Downloads\Accel_mode.xlsx"
book = load_workbook(output_excel)

# เพิ่มข้อมูลลงในไฟล์ Excel
with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
    writer.book = book
    
    # Append DataFrame ลงใน Excel sheet ที่มีอยู่แล้ว
    df.to_excel(writer, sheet_name='SKU_Serials', index=False)

    writer.save()

print(f"ข้อมูลถูกเพิ่มลงใน {output_excel} เรียบร้อยแล้ว")

บริษัท ไอที ซิตี้ จำกัด (มหาชน)
ใบโอนสินค้า
From Branchวันที่
ผู้บันทึกTo Branch
ผู้ส่งสินค้าออก หมายเหตุ7/8/24 10:48 AM
Shopee X KERRYExpress
PR CannonTRB018324070800041 Request_No.
ณัฏฐชัยนันท์  วิจักษ์รัตนากรB0183 WDCR1 - IT CITY warehouse Samrong
B0183 WITO1 - IT CITY OnlineV2.8
 
No. Product Code Barcode Product Name Transfer No. Order Ship Status
1PR1-000550 CANON PIXMA IX6770 (1Y) 5 5Shipped
 
 
918748B01292AB21AGWH12479, 918748B01292AB21AGWH12475, 918748B01292AB21AGWH12441,
918748B01292AB21AGWH12114, 918748B01292AB21AGWH12123Serial :
2PR1-000551 CANON Photo IX6870 (1Y) 2 2Shipped
 
 
918747B01292AB21AGXA01956, 918747B01292AB21AGXA01962Serial :
3PR1-000584 CANON Pixma TS307 (1Y) 1 1Shipped
 
 
912321C03292BA21AGYE13786Serial :
4PR1-000585 CANON PIXMA G1010 (2Y) 1 1Shipped
 
 
912314C01292AB21KPGJ09924Serial :
5PR2-000495 CANON LASER SHOT LBP6030W (1Y) 10 10Shipped
 
 
918468B00992AA21NTMA656500, 918468B00992AA21NTMA656704, 918468B00992AA21NTMA656507,
918468B00992AA21NTMA656475, 

IndexError: At least one sheet must be visible